# Technical Test - JLL Technologies

### Problem Statement : 
You are part of a team of 20 analysts who would like to give 2 hours each per week to support a philanthropic cause.We have dataset that lists 50k+ registered charities in Australia.
##### Problem is where to invest their time to generate the maximum value for society ?

###### As per the initial glance at the data, following are the key points I'm focusing:
- Charity cause (who they help, what they do)
- Impact or community need
- Operational scale or challenges
- Areas where skilled support is needed

This is an approach to clean the various data collected from ACNC website and keep only relevant information that we can use as a data source for tableau visualization. The links to the data sources are provided in the PPT.

## Data Cleaning

In [1]:
import pandas as pd
pd.set_option('display.max_rows', None)  # Show all rows
pd.set_option('display.max_columns', None) # Show all columns


In [2]:
#Import All necessary data 
# Replace 'your_file.xlsx' with the path to your Excel file

ais20_og = pd.read_excel('datadotgov_ais20.xlsx',sheet_name='Sheet1')
programs_og = pd.read_excel('datadotgov_ais20_programs.xlsx',sheet_name='Sheet1')
main_og = pd.read_excel('datadotgov_main.xlsx',sheet_name='Sheet1')


We are going to analyse each dataset and we will clean up the data.

In [4]:
ais20 = ais20_og  #I'm storing ais20_og to a new variable for data manipulation

In [5]:
ais20.shape # To identify the data structure

(51092, 93)

ais20 has 93 columns and 51092 records

In [6]:
ais20.isnull().sum()

abn                                                                                  0
charity name                                                                         0
registration status                                                                  0
charity website                                                                  17053
charity size                                                                         6
basic religious charity                                                              0
ais due date                                                                        14
date ais received                                                                    9
financial report date received                                                   19499
conducted activities                                                                 0
why charity did not conduct activities                                           48851
international activities details           

Many columns have huge amount of rows that has NULLS. This needs to be carefully reviewed and if not required we can get rid of them.

Check the list of columns to identify if we have anything to exclude on the first glance keeping the goal in mind

In [7]:
#check the columns in the datasets
ais20.columns.tolist()

['abn',
 'charity name',
 'registration status',
 'charity website',
 'charity size',
 'basic religious charity',
 'ais due date',
 'date ais received',
 'financial report date received',
 'conducted activities',
 'why charity did not conduct activities',
 'international activities details',
 'international activities undertaken - transferring goods or services overseas',
 'international activities undertaken - operating overseas including programs',
 'other international activities',
 'other international activities description',
 'how purposes were pursued',
 'staff - full time',
 'staff - part time',
 'staff - casual',
 'total full time equivalent staff',
 'staff - volunteers',
 'cash or accrual',
 'type of financial statement',
 'report consolidated with more than one entity',
 'charity report has a modification',
 'type of report modification',
 'charity has related party transactions',
 'charity has documented policies for related party transactions',
 'fin report from',
 'fin re

There are few columns that we can exclude, as those doesn't align much with my focus points

In [8]:
# Drop columns containing 'temp' or 'debug' or 'fundraising' as i don't find them useful keepoing the goal in mind
ais20 = ais20.loc[:, ~ais20.columns.str.contains('submitted |association|fundraising')]


In [9]:
# Find columns with more than 70% nulls
null_cols = ais20.columns[ais20.isnull().sum() > 35000]
len(null_cols) # No.of columns with more than 70% NULLS

9

In [7]:
# Print unique values for those columns
for col in null_cols:
    print(f"\n--- Unique values in '{col}' (NaNs: {ais20[col].isnull().sum()}) ---")
    print(ais20[col].dropna().unique())



--- Unique values in 'why charity did not conduct activities' (NaNs: 48851) ---
['Yagoona Baptist Church has now closed and a Curch plant has been made under the name of Graceway Church.'
 'The operations and programs of the charity are undertaken by Save the Children Australia.'
 'Because of COVIC19 we were unable to conduct our usual activities' ...
 'Although, we became a registered charity on 15 November 2019, the operational services provided by WOW to the Cairns community was funded by Stratford Medical Centre until 30 June 2020, as we did not receive funding to operate as our own entity until 1 July 2020. From November 2019, we did fundraise for our fit-for-purpose mobile van.'
 'Planning and development is still in progress'
 'Baptist Relief Fund did not do any emergency relief work last year due to no disaster events.']

--- Unique values in 'international activities details' (NaNs: 48665) ---
['AMT receives gifts from churches and individuals and distributes these monies in 

In [10]:
#I'm excluding columns with large text description here as it's difficult to read and understand from here.
ExCol = ['why charity did not conduct activities','international activities details','other international activities description']
null_cols = [col for col in null_cols if col not in ExCol]

In [9]:
for col in null_cols:
    print(f"\n--- Unique values in '{col}' (NaNs: {ais20[col].isnull().sum()}) ---")
    print(ais20[col].dropna().unique())



--- Unique values in 'type of financial statement' (NaNs: 35321) ---
['General purpose financial statements'
 'Special purpose financial statements'
 'General purpose financial statements – reduced disclosure regime'
 'General purpose financial statements – simplified disclosure'
 'General purpose financial statements   reduced disclosure regime'
 'General purpose financial statements - reduced disclosure regime'
 'General purpose financial statements   simplified disclosure'
 'Special purpose financial statement'
 'General purpose financial statements - simplified disclosure']

--- Unique values in 'report consolidated with more than one entity' (NaNs: 35324) ---
['n' 'y']

--- Unique values in 'type of report modification' (NaNs: 49488) ---
['Qualified/qualification' 'Disclaimed/disclaimer' 'Adverse']

--- Unique values in 'charity has related party transactions' (NaNs: 35348) ---
['n' 'y']

--- Unique values in 'charity has documented policies for related party transactions' (NaNs:

Analysing the values, we can use it for in-depth analysis like Why the report fot modified? How the related party transactions and allied documents are modified and managed. But this will direct us more towards the robustness of the charity or how well established the charities are. We cannot get a direct information about the need of support from these fields.

So let's keep them away from this analysis.

In [11]:
# Drop them from the DataFrame
ais20 = ais20.drop(columns=null_cols)
ais20.shape

(51092, 54)

In [11]:
#check again the columns in the datasets
ais20.columns.tolist()

['abn',
 'charity name',
 'registration status',
 'charity website',
 'charity size',
 'basic religious charity',
 'ais due date',
 'date ais received',
 'financial report date received',
 'conducted activities',
 'why charity did not conduct activities',
 'international activities details',
 'international activities undertaken - transferring goods or services overseas',
 'international activities undertaken - operating overseas including programs',
 'other international activities',
 'other international activities description',
 'how purposes were pursued',
 'staff - full time',
 'staff - part time',
 'staff - casual',
 'total full time equivalent staff',
 'staff - volunteers',
 'cash or accrual',
 'charity report has a modification',
 'fin report from',
 'fin report to',
 'revenue from government',
 'donations and bequests',
 'revenue from goods and services',
 'revenue from investments',
 'all other revenue',
 'total revenue',
 'other income',
 'total gross income',
 'employee exp

Going through these columns and it's values, many columns are not in alignment with our goal. So I have manually picked them up and removing here.

In [12]:
# List of columns to drop
columns_to_drop = [
    'charity website',# Not relevant for our goal
    'ais due date',  # Shows report submission deadline, not relevant for impact analysis
    'date ais received',  # Tracking of when report was received; does not affect charity operations
    'financial report date received',  # Internal ACNC processing date; not useful for prioritizing support
    'charity report has a modification',  # Audit mistakes; doesn't relate to direct need or service delivery
    'charity report has a modification',  # Not really useful for our goal
]

# Drop them from ais20
ais20 = ais20.drop(columns=columns_to_drop)


In [13]:
ais20.shape

(51092, 49)

In [13]:
#check again the columns in the datasets
ais20.columns.tolist()

['abn',
 'charity name',
 'registration status',
 'charity size',
 'basic religious charity',
 'conducted activities',
 'why charity did not conduct activities',
 'international activities details',
 'international activities undertaken - transferring goods or services overseas',
 'international activities undertaken - operating overseas including programs',
 'other international activities',
 'other international activities description',
 'how purposes were pursued',
 'staff - full time',
 'staff - part time',
 'staff - casual',
 'total full time equivalent staff',
 'staff - volunteers',
 'cash or accrual',
 'fin report from',
 'fin report to',
 'revenue from government',
 'donations and bequests',
 'revenue from goods and services',
 'revenue from investments',
 'all other revenue',
 'total revenue',
 'other income',
 'total gross income',
 'employee expenses',
 'interest expenses',
 'grants and donations made for use in Australia',
 'grants and donations made for use outside Austral

As per the ACNC Explanatory notes, charities with ABN number like '911111111xx' are reporting groups.We can use this as a filter to limit our data.

In [14]:
count = ais20['abn'].astype(str).str.startswith('911111111').sum()
print("Number of ABNs starting with '911111111':", count)
ais20.shape

Number of ABNs starting with '911111111': 62


(51092, 49)

In [15]:
#Remove records with ABN number like '911111111xx' 
ais20 = ais20[~ais20['abn'].astype(str).str.startswith('911111111')]
ais20.shape

(51030, 49)

In [16]:
# Check the registration status column

ais20['registration status'].unique()

array(['Registered', 'Voluntarily Revoked No Longer Operating',
       'Revoked Double Defaulter', 'Voluntarily Revoked Merged',
       'Voluntarily Revoked Not Entitled To Be A Charity',
       'Voluntarily Revoked Continuing But No Longer Wants To Be A Charity',
       'Revoked ABN', 'Revoked RTS', 'Revoked Registration',
       'Revoked Compliance'], dtype=object)

This shows us the unique registration status.We should be considering only those with 'Resgistered' value, as the rest are not in alignment with our goal. 

In [16]:
# Check the count for only 'Registered' status
RegCharities = (ais20['registration status'] == 'Registered').sum()
print(f"Number of charities in 'Registered' status: {RegCharities}")


Number of charities in 'Registered' status: 46214


In [17]:
# Filter the data for only 'Registered' status
ais20_final = ais20[ais20['registration status'] == 'Registered']


In [18]:
ais20_final.shape

(46214, 49)

In [20]:
ais20_final.head()

,abn,charity name,registration status,charity size,basic religious charity,conducted activities,why charity did not conduct activities,international activities details,international activities undertaken - transferring goods or services overseas,international activities undertaken - operating overseas including programs,other international activities,other international activities description,how purposes were pursued,staff - full time,staff - part time,staff - casual,total full time equivalent staff,staff - volunteers,cash or accrual,fin report from,fin report to,revenue from government,donations and bequests,revenue from goods and services,revenue from investments,all other revenue,total revenue,other income,total gross income,employee expenses,interest expenses,grants and donations made for use in Australia,grants and donations made for use outside Australia,all other expenses,total expenses,net surplus/deficit,other comprehensive income,total comprehensive income,total current assets,non-current loans receivable,other non-current assets,total non-current assets,total assets,total current liabilities,non-current loans payable,other non-current liabilities,total non-current liabilities,total liabilities,net assets/liabilities
0,11000047950,Sydney Missionary & Bible College,Registered,Large,n,y,NaN,NaN,n,n,n,NaN,"Conducting an interdenominational, community b...",37,11,20,56.45,3,NaN,01/01/2020,31/12/2020,1153000,1004051,5912764,3975,1668,8075458,35452,8110910,3178458,3307,336602,0,3404370,6922737,1188173,0,1188173,3689978,0,30993759,30993759,34683737,925378,70000,82089,152089,1077467,33606270
1,11000073870,Integricare Limited,Registered,Large,n,y,NaN,NaN,n,n,n,NaN,We are an early education provider and also of...,120,80,25,150.00,3,NaN,01/07/2019,30/06/2020,6271343,85250,7823475,574470,105745,14860283,2988,14863271,11251713,0,0,0,3282691,14534404,328867,0,328867,6602193,0,12546393,12546393,19148586,2568965,0,209735,209735,2778700,16369886
2,11000761571,AUSTRALIANS IN MISSION TOGETHER,Registered,Large,n,y,NaN,AMT receives gifts from churches and individua...,y,n,n,NaN,AMT served the needs of volunteer workers in c...,2,2,0,3.20,1,NaN,01/01/2020,31/12/2020,0,2537967,51715,93486,574396,3257564,0,3257564,176773,0,0,3266654,156676,3600103,-342539,0,-342539,4939229,0,3451441,3451441,8390670,593396,0,336,336,593732,7796938
3,11001233790,The Greek Orthodox Church & Community Of The H...,Registered,Medium,n,y,NaN,NaN,n,n,n,NaN,Operations of Religious activities through th...,1,0,0,1.00,50,NaN,01/07/2019,30/06/2020,15869,54722,177947,50107,52231,350876,0,350876,62755,0,0,0,234566,297321,53555,0,53555,42423,0,0,2058485,2100908,2705,0,0,325481,328186,1772722
4,11001241005,Wangarang Industries Limited,Registered,Large,n,y,NaN,NaN,n,n,n,NaN,1. Wangarang is an Australian Supported Employ...,42,123,11,105.00,0,NaN,01/07/2019,30/06/2020,2282916,9218,1978569,13493,51638,4335834,1364,4337198,3353501,2788,0,0,779912,4136201,200997,0,200997,1646148,0,2584562,2584562,4230710,1014463,0,64289,64289,1078752,3151958


In [19]:
ais20_final = ais20_final.drop(columns='registration status')


In [20]:
ais20_final.shape

(46214, 48)

### ais20 has been cleaned and many irrelevant/least relevant columns are removed and result is stored in ais20_final variable.

### Let's play around with second dataset :  programs_og

In [21]:
programs = programs_og  #I'm storing programs_og to a new variable for data manipulation

In [22]:
programs.shape # To identify the data structure

(75757, 58)

This dataset has 58 columns and 75757 rows.

In [23]:
programs.isnull().sum()

ABN                                                                     0
Charity Name                                                            0
Registration Status                                                     0
Program name                                                            9
Classification                                                          0
Children - aged 6 to under 15                                           0
Environment                                                             0
Families                                                                0
General community in Australia                                          0
Migrants, refugees or asylum seekers                                    0
Overseas communities or charities                                       0
Aboriginal and Torres Strait Islander people                            0
Adults - aged 65 and over                                               0
Early childhood - aged under 6        

So many columns have lots of NULLS. We need to inspect and remove those if found to be irrelevant.

In [24]:
# Drop columns containing 'Operating Location' as i don't find them useful keepoing the goal in mind
programs = programs.loc[:, ~programs.columns.str.contains('Operating Location')]

In [25]:
# Find columns with more than 70% nulls
null_cols2 = programs.columns[programs.isnull().sum() > 49000]
len(null_cols2) # No.of columns with more than 70% NULLS

2

In [26]:
# Print unique values for those columns
for col in null_cols2:
    print(f"\n--- Unique values in '{col}' (NaNs: {programs[col].isnull().sum()}) ---")
    print(programs[col].dropna().unique())



--- Unique values in 'other description' (NaNs: 73324) ---
['General community in Australia' 'Adults aged 55 and over'
 'People with mental health illness' ...
 'General communities in Australia' 'Parish of All Souls Anglican Church'
 'we support our missionaries in Australia and other parts of the world.']

--- Unique values in 'overseas countries' (NaNs: 71391) ---
['Brazil;Chad;Colombia;Congo;Fiji;France;Hong Kong;India;Italy;Mexico;Nepal;Netherlands;Papua New Guinea;Philippines;Portugal;South Africa;Spain;Thailand;Timor-Leste;Uganda;United States;Venezuela, Bolivarian Republic of;Zambia'
 'Fiji;Hong Kong;Singapore;United Arab Emirates'
 'Germany;United Kingdom;United States' ...
 'Canada;New Zealand;Singapore;South Africa;United Kingdom;United States'
 'Canada;New Zealand;United Kingdom;United States;Vanuatu'
 'Benin;France;Kenya;Niger;Papua New Guinea;Portugal;Thailand']


Let's drop these for now as I don't see them adding any insight with huge NULLS in it. Also let's drop Charity weblink as it's not useful in the current context.

In [26]:
null_cols2 = list(null_cols2)
null_cols2.append('Charity weblink')
null_cols2

['other description', 'overseas countries', 'Charity weblink']

In [27]:
programs = programs.drop(columns=null_cols2)

As per the ACNC Explanatory notes, charities with ABN number like '911111111xx' are reporting groups.We can use this as a filter to limit our data.

In [28]:
count = programs['ABN'].astype(str).str.startswith('911111111').sum()
print("Number of ABNs starting with '911111111':", count)
programs.shape

Number of ABNs starting with '911111111': 202


(75757, 35)

In [29]:
#Remove records with ABN number like '911111111xx' 
programs = programs[~programs['ABN'].astype(str).str.startswith('911111111')]
programs.shape

(75555, 35)

In [30]:
# Check the count for only 'Registered' status
RegCharities = (programs['Registration Status'] == 'Registered').sum()
print(f"Number of charities in 'Registered' status: {RegCharities}")

Number of charities in 'Registered' status: 70310


In [31]:
# Filter the data for only 'Registered' status
programs = programs[programs['Registration Status'] == 'Registered']


(70310, 35)

In [32]:
programs = programs.drop(columns='Registration Status')
programs.shape

(70310, 34)

In [33]:
#check again the columns in the datasets
programs.columns.tolist()

['ABN',
 'Charity Name',
 'Program name',
 'Classification',
 'Children - aged 6 to under 15',
 'Environment',
 'Families',
 'General community in Australia',
 'Migrants, refugees or asylum seekers',
 'Overseas communities or charities',
 'Aboriginal and Torres Strait Islander people',
 'Adults - aged 65 and over',
 'Early childhood - aged under 6',
 'Females',
 'Gay, lesbian, bisexual, transgender or intersex persons',
 'Males',
 'People at risk of homelessness/ people experiencing homelessness',
 'People with disabilities',
 'Victims of crime (including family violence)',
 'Animals',
 'Financially disadvantaged people',
 'People in rural/regional/remote communities',
 'People with chronic illness (including terminal illness)',
 'Pre/post release offenders and/or their families',
 'Veterans and/or their families',
 'Youth - 15 to under 25',
 'Adults - aged 25 to under 65',
 'Other charities',
 'People from a culturally and linguistically diverse background',
 'Unemployed persons',
 'V

'Children - aged 6 to under 15',
 'Environment',
 'Families',
 'General community in Australia',
 'Migrants, refugees or asylum seekers',
 'Overseas communities or charities',
 'Aboriginal and Torres Strait Islander people',
 'Adults - aged 65 and over',
 'Early childhood - aged under 6',
 'Females',
 'Gay, lesbian, bisexual, transgender or intersex persons',
 'Males',
 'People at risk of homelessness/ people experiencing homelessness',
 'People with disabilities',
 'Victims of crime (including family violence)',
 'Animals',
 'Financially disadvantaged people',
 'People in rural/regional/remote communities',
 'People with chronic illness (including terminal illness)',
 'Pre/post release offenders and/or their families',
 'Veterans and/or their families',
 'Youth - 15 to under 25',
 'Adults - aged 25 to under 65',
 'Other charities',
 'People from a culturally and linguistically diverse background',
 'Unemployed persons',
 'Victims of disaster',
 'Other',
 'other description'
 
 #### these columns are the 'Beneficiaries'. Let's transpose these columns, which will make the tableau operations easier

In [34]:
# Let's create beneficiaries dataset for each Charity
Cols_To_Transform = ['ABN',
 'Children - aged 6 to under 15',
 'Environment',
 'Families',
 'General community in Australia',
 'Migrants, refugees or asylum seekers',
 'Overseas communities or charities',
 'Aboriginal and Torres Strait Islander people',
 'Adults - aged 65 and over',
 'Early childhood - aged under 6',
 'Females',
 'Gay, lesbian, bisexual, transgender or intersex persons',
 'Males',
 'People at risk of homelessness/ people experiencing homelessness',
 'People with disabilities',
 'Victims of crime (including family violence)',
 'Animals',
 'Financially disadvantaged people',
 'People in rural/regional/remote communities',
 'People with chronic illness (including terminal illness)',
 'Pre/post release offenders and/or their families',
 'Veterans and/or their families',
 'Youth - 15 to under 25',
 'Adults - aged 25 to under 65',
 'Other charities',
 'People from a culturally and linguistically diverse background',
 'Unemployed persons',
 'Victims of disaster',
 'Other' ]

In [35]:
# Transpose the data
benef = programs[Cols_To_Transform].set_index('ABN').stack().reset_index().rename(columns={'level_1': 'Beneficiary Type', 0: 'Value'})
# Filter only rows where Value is 'Y'
benef = benef[benef['Value'] == 'Y']

In [37]:
benef = benef.drop(columns='Value')
benef.head(10)

,ABN,Beneficiary Type
3,11000047950,General community in Australia
5,11000047950,Overseas communities or charities
17,11000047950,People in rural/regional/remote communities
22,11000047950,Adults - aged 25 to under 65
24,11000047950,People from a culturally and linguistically di...
30,11000073870,Families
34,11000073870,Aboriginal and Torres Strait Islander people
36,11000073870,Early childhood - aged under 6
58,11000073870,Families
62,11000073870,Aboriginal and Torres Strait Islander people


Now the 'benef' variable contains dataset that has information about Charities and their respective beneficiaries.

I can remove these transformed columns now from the original 'program' set and join back 'benef' to 'program' to get the data at a single place.

In [38]:
# Remove 'ABN' from the list since that's the key field
Cols_To_Transform.remove('ABN')

# Drop the remaining columns
programs = programs.drop(columns=Cols_To_Transform, errors='ignore')


In [211]:
programs.head()

,ABN,Charity Name,Registration Status,Program name,Classification,Operating online,Operating overseas
0,11000047950,Sydney Missionary & Bible College,Registered,Sydney Missionary & Bible College,Continuing education,N,N
1,11000073870,Integricare Limited,Registered,Supported Playgroups - North,Playgroups,N,N
2,11000073870,Integricare Limited,Registered,Early Education Centres,Early childhood education,N,N
3,11000073870,Integricare Limited,Registered,Supported Playgroups - South,Playgroups,N,N
4,11000761571,AUSTRALIANS IN MISSION TOGETHER,Registered,Worldwide Mission,Christianity,N,Y


Let's JOIN the 2 'benef' and 'programs' datasets to get combined information in single place.

In [39]:
programs_final = programs.merge(benef, how='left', on='ABN')

In [40]:
programs_final.shape

(1062215, 7)

In [41]:
programs_final.head()

,ABN,Charity Name,Program name,Classification,Operating online,Operating overseas,Beneficiary Type
0,11000047950,Sydney Missionary & Bible College,Sydney Missionary & Bible College,Continuing education,N,N,General community in Australia
1,11000047950,Sydney Missionary & Bible College,Sydney Missionary & Bible College,Continuing education,N,N,Overseas communities or charities
2,11000047950,Sydney Missionary & Bible College,Sydney Missionary & Bible College,Continuing education,N,N,People in rural/regional/remote communities
3,11000047950,Sydney Missionary & Bible College,Sydney Missionary & Bible College,Continuing education,N,N,Adults - aged 25 to under 65
4,11000047950,Sydney Missionary & Bible College,Sydney Missionary & Bible College,Continuing education,N,N,People from a culturally and linguistically di...


### programs has been cleaned and many irrelevant/least relevant columns are removed and result is stored in programs_final variable.

### Let's dive into the third data set 

In [78]:
main = main_og

In [79]:
main.shape

(63184, 69)

In [80]:
main.isnull().sum()

ABN                                                                 473
Charity_Legal_Name                                                  301
Other_Organisation_Names                                          57382
Address_Type                                                          0
Address_Line_1                                                     6797
Address_Line_2                                                    57853
Address_Line_3                                                    62525
Town_City                                                          6782
State                                                              6847
Postcode                                                           6710
Country                                                            7833
Charity_Website                                                   22027
Registration_Date                                                   341
Date_Organisation_Established                                   

It is observed that our key column 'ABN' has NULLS, we might need to get rid of those, as those records won't be able to join back to our existing dataset.

In [81]:
main = main[main['ABN'].notna()] #Removes rows that have NULLS for key field ABN
# Drop columns containing 'Address' as i don't find them useful keepoing the goal in mind
main = main.loc[:, ~main.columns.str.contains('Address')] #Removes unnecessary address level info
#ABN was treated as float, convert to INT
main['ABN'] = main['ABN'].astype('Int64')  # Int64 handles NaN values as well
main = main.drop(columns=['Charity_Website','Charity_Legal_Name','Other_Organisation_Names'])



In [82]:
count = main['ABN'].astype(str).str.startswith('911111111').sum()
print("Number of ABNs starting with '911111111':", count)
main.shape

Number of ABNs starting with '911111111': 0


(62711, 62)

In [83]:
#Remove records with ABN number like '911111111xx' if any 
main = main[~main['ABN'].astype(str).str.startswith('911111111')]
main.shape

(62711, 62)

In [84]:
main.columns.tolist()

['ABN',
 'Town_City',
 'State',
 'Postcode',
 'Country',
 'Registration_Date',
 'Date_Organisation_Established',
 'Charity_Size',
 'Number_of_Responsible_Persons',
 'Financial_Year_End',
 'Operates_in_ACT',
 'Operates_in_NSW',
 'Operates_in_NT',
 'Operates_in_QLD',
 'Operates_in_SA',
 'Operates_in_TAS',
 'Operates_in_VIC',
 'Operates_in_WA',
 'Operating_Countries',
 'PBI',
 'HPC',
 'Preventing_or_relieving_suffering_of_animals',
 'Advancing_Culture',
 'Advancing_Education',
 'Advancing_Health',
 'Promote_or_oppose_a_change_to_law__government_poll_or_prac',
 'Advancing_natual_environment',
 'Promoting_or_protecting_human_rights',
 'Purposes_beneficial_to_ther_general_public_and_other_analogous',
 'Promoting_reconciliation__mutual_respect_and_tolerance',
 'Advancing_Religion',
 'Advancing_social_or_public_welfare',
 'Advancing_security_or_safety_of_Australia_or_Australian_public',
 'Aboriginal_or_TSI',
 'Adults',
 'Aged_Persons',
 'Children',
 'Communities_Overseas',
 'Early_Childhood',


In [85]:
#Transform the states to get Operating states info
States_To_Transform = [
 'ABN',  
 'Operates_in_ACT',
 'Operates_in_NSW',
 'Operates_in_NT',
 'Operates_in_QLD',
 'Operates_in_SA',
 'Operates_in_TAS',
 'Operates_in_VIC',
 'Operates_in_WA'   
]

In [86]:
# Transpose the data
OperationalState = main[States_To_Transform].set_index('ABN').stack().reset_index().rename(columns={'level_1': 'Operating State', 0: 'Value'})
# Filter only rows where Value is 'Y'
OperationalState = OperationalState[OperationalState['Value'] == 'Y']

In [87]:
# Remove 'Operates_in_' from values in the 'Operating State' column
OperationalState['Operating State'] = OperationalState['Operating State'].str.replace('Operates_in_', '', regex=False)



In [88]:
OperationalState = OperationalState.drop(columns='Value')
OperationalState.head(10)

,ABN,Operating State
0,51214424410,SA
1,62601233489,ACT
2,62601233489,NSW
3,62601233489,NT
4,62601233489,QLD
5,62601233489,SA
6,62601233489,TAS
7,62601233489,VIC
8,62601233489,WA
9,18246792704,NSW


In [89]:
# Remove 'ABN' from the list since that's the key field
States_To_Transform.remove('ABN')

# Drop the remaining columns
main = main.drop(columns=States_To_Transform, errors='ignore')


In [90]:
SubTypes_To_Transform = [
 'ABN',
 'PBI',
 'HPC',
 'Preventing_or_relieving_suffering_of_animals',
 'Advancing_Culture',
 'Advancing_Education',
 'Advancing_Health',
 'Promote_or_oppose_a_change_to_law__government_poll_or_prac',
 'Advancing_natual_environment',
 'Promoting_or_protecting_human_rights',
 'Purposes_beneficial_to_ther_general_public_and_other_analogous',
 'Promoting_reconciliation__mutual_respect_and_tolerance',
 'Advancing_Religion',
 'Advancing_social_or_public_welfare',
 'Advancing_security_or_safety_of_Australia_or_Australian_public'
]

In [91]:
# Transpose the data
SubType = main[SubTypes_To_Transform].set_index('ABN').stack().reset_index().rename(columns={'level_1': 'Sub Type', 0: 'Value'})
# Filter only rows where Value is 'Y'
SubType = SubType[SubType['Value'] == 'Y']

In [92]:
SubType = SubType.drop(columns='Value')
SubType.head()

,ABN,Sub Type
0,51214424410,Advancing_Education
1,62601233489,Advancing_Religion
2,18246792704,Advancing_natual_environment
3,64423343545,Advancing_Religion
4,28650300600,Advancing_Education


In [93]:
# Remove 'ABN' from the list since that's the key field
SubTypes_To_Transform.remove('ABN')

# Drop the remaining columns
main = main.drop(columns=SubTypes_To_Transform, errors='ignore')

In [94]:
main.columns.tolist()

['ABN',
 'Town_City',
 'State',
 'Postcode',
 'Country',
 'Registration_Date',
 'Date_Organisation_Established',
 'Charity_Size',
 'Number_of_Responsible_Persons',
 'Financial_Year_End',
 'Operating_Countries',
 'Aboriginal_or_TSI',
 'Adults',
 'Aged_Persons',
 'Children',
 'Communities_Overseas',
 'Early_Childhood',
 'Ethnic_Groups',
 'Families',
 'Females',
 'Financially_Disadvantaged',
 'LGBTIQA+',
 'General_Community_in_Australia',
 'Males',
 'Migrants_Refugees_or_Asylum_Seekers',
 'Other_Beneficiaries',
 'Other_Charities',
 'People_at_risk_of_homelessness',
 'People_with_Chronic_Illness',
 'People_with_Disabilities',
 'Pre_Post_Release_Offenders',
 'Rural_Regional_Remote_Communities',
 'Unemployed_Person',
 'Veterans_or_their_families',
 'Victims_of_crime',
 'Victims_of_Disasters',
 'Youth',
 'animals',
 'environment',
 'other_gender_identities']

In [95]:
Benef_To_Transform = [
 'ABN',
 'Aboriginal_or_TSI',
 'Adults',
 'Aged_Persons',
 'Children',
 'Communities_Overseas',
 'Early_Childhood',
 'Ethnic_Groups',
 'Families',
 'Females',
 'Financially_Disadvantaged',
 'LGBTIQA+',
 'General_Community_in_Australia',
 'Males',
 'Migrants_Refugees_or_Asylum_Seekers',
 'Other_Beneficiaries',
 'Other_Charities',
 'People_at_risk_of_homelessness',
 'People_with_Chronic_Illness',
 'People_with_Disabilities',
 'Pre_Post_Release_Offenders',
 'Rural_Regional_Remote_Communities',
 'Unemployed_Person',
 'Veterans_or_their_families',
 'Victims_of_crime',
 'Victims_of_Disasters',
 'Youth',
 'animals',
 'environment',
 'other_gender_identities'
]

In [96]:
# Transpose the data
Beneficiary = main[Benef_To_Transform].set_index('ABN').stack().reset_index().rename(columns={'level_1': 'Beneficiary', 0: 'Value'})
# Filter only rows where Value is 'Y'
Beneficiary = Beneficiary[Beneficiary['Value'] == 'Y']

In [97]:
Beneficiary = Beneficiary.drop(columns='Value')
Beneficiary.head(20)

,ABN,Beneficiary
0,51214424410,Aged_Persons
1,51214424410,General_Community_in_Australia
2,62601233489,Females
3,18246792704,Aboriginal_or_TSI
4,18246792704,Adults
5,18246792704,Aged_Persons
6,18246792704,Children
7,18246792704,Early_Childhood
8,18246792704,Ethnic_Groups
9,18246792704,Families


Beneficiary info is already available with us in the above table(programs). So this will be a repetitive info for us while joining these 2 datasets together. So let's not keep these columns.

In [98]:
# Remove 'ABN' from the list since that's the key field
Benef_To_Transform.remove('ABN')

# Drop the remaining columns
main = main.drop(columns=Benef_To_Transform, errors='ignore')

In [100]:
main.head()
main.shape

(62711, 11)

In [101]:
# Left join 'main' with 'SubType' on 'ABN'
main_final = main.merge(SubType, on='ABN', how='left')

## Left join the result with 'Beneficiary' on 'ABN' 
#main_final = main_final.merge(Beneficiary, on='ABN', how='left') --Considered this initially, but this increases the redundancy

# Left join the result with 'OperationalState' on 'ABN'
main_final = main_final.merge(OperationalState, on='ABN', how='left')


In [110]:
main_final.shape

(109231, 13)

In [111]:
main_final.head(20)

,ABN,Town_City,State,Postcode,Country,Registration_Date,Date_Organisation_Established,Charity_Size,Number_of_Responsible_Persons,Financial_Year_End,Operating_Countries,Sub Type,Operating State
0,51214424410,Enfield,SA,5085,Australia,20/02/2018,20/02/2018,NaN,9,30-Jun,NaN,Advancing_Education,SA
1,62601233489,Briar Hill,VIC,3088,Australia,01/07/2023,09/09/2014,NaN,10,30-Jun,NaN,Advancing_Religion,ACT
2,62601233489,Briar Hill,VIC,3088,Australia,01/07/2023,09/09/2014,NaN,10,30-Jun,NaN,Advancing_Religion,NSW
3,62601233489,Briar Hill,VIC,3088,Australia,01/07/2023,09/09/2014,NaN,10,30-Jun,NaN,Advancing_Religion,NT
4,62601233489,Briar Hill,VIC,3088,Australia,01/07/2023,09/09/2014,NaN,10,30-Jun,NaN,Advancing_Religion,QLD
5,62601233489,Briar Hill,VIC,3088,Australia,01/07/2023,09/09/2014,NaN,10,30-Jun,NaN,Advancing_Religion,SA
6,62601233489,Briar Hill,VIC,3088,Australia,01/07/2023,09/09/2014,NaN,10,30-Jun,NaN,Advancing_Religion,TAS
7,62601233489,Briar Hill,VIC,3088,Australia,01/07/2023,09/09/2014,NaN,10,30-Jun,NaN,Advancing_Religion,VIC
8,62601233489,Briar Hill,VIC,3088,Australia,01/07/2023,09/09/2014,NaN,10,30-Jun,NaN,Advancing_Religion,WA
9,18246792704,Liverpool BC,NSW,1871,Australia,27/04/2017,20/11/1997,NaN,10,30-Jun,NaN,Advancing_natual_environment,NSW


We have our main dataset also ready. Now we have to Join all 3 datasets and use that 'Combined' data for visualisation and further analysis

### Combining 'ais20_final','Programs_final' and 'main_final'

Since I'm focusing on data from all the 3 datasets, I'll be doing an INNER JOIN here. I'm going to continue my analysis on the Charities that are available in all the 3 datasets.

ais20_final has 'abn' as the key column name. We need to make that same for all three datsets.

In [106]:
ais20_final.rename(columns={'abn': 'ABN'}, inplace=True)

In [107]:
# Inner join 'ais20' with 'program' on 'ABN'
finaldata = ais20_final.merge(programs_final, on='ABN', how='inner')

# Inner join 'main' with 'finaldata' on 'ABN'
finaldata = finaldata.merge(main_final, on='ABN', how='inner')

finaldata.shape


(2161617, 66)

In [109]:
finaldata.shape

(2161617, 66)

In [112]:
finaldata.head(20)

,ABN,charity name,charity size,basic religious charity,conducted activities,why charity did not conduct activities,international activities details,international activities undertaken - transferring goods or services overseas,international activities undertaken - operating overseas including programs,other international activities,other international activities description,how purposes were pursued,staff - full time,staff - part time,staff - casual,total full time equivalent staff,staff - volunteers,cash or accrual,fin report from,fin report to,revenue from government,donations and bequests,revenue from goods and services,revenue from investments,all other revenue,total revenue,other income,total gross income,employee expenses,interest expenses,grants and donations made for use in Australia,grants and donations made for use outside Australia,all other expenses,total expenses,net surplus/deficit,other comprehensive income,total comprehensive income,total current assets,non-current loans receivable,other non-current assets,total non-current assets,total assets,total current liabilities,non-current loans payable,other non-current liabilities,total non-current liabilities,total liabilities,net assets/liabilities,Charity Name,Program name,Classification,Operating online,Operating overseas,Beneficiary Type,Town_City,State,Postcode,Country,Registration_Date,Date_Organisation_Established,Charity_Size,Number_of_Responsible_Persons,Financial_Year_End,Operating_Countries,Sub Type,Operating State
0,11000047950,Sydney Missionary & Bible College,Large,n,y,NaN,NaN,n,n,n,NaN,"Conducting an interdenominational, community b...",37,11,20,56.45,3,NaN,01/01/2020,31/12/2020,1153000,1004051,5912764,3975,1668,8075458,35452,8110910,3178458,3307,336602,0,3404370,6922737,1188173,0,1188173,3689978,0,30993759,30993759,34683737,925378,70000,82089,152089,1077467,33606270,Sydney Missionary & Bible College,Sydney Missionary & Bible College,Continuing education,N,N,General community in Australia,Croydon,NSW,2132,Australia,03/12/2012,NaN,Large,9,31-Dec,NaN,Advancing_Education,NSW
1,11000047950,Sydney Missionary & Bible College,Large,n,y,NaN,NaN,n,n,n,NaN,"Conducting an interdenominational, community b...",37,11,20,56.45,3,NaN,01/01/2020,31/12/2020,1153000,1004051,5912764,3975,1668,8075458,35452,8110910,3178458,3307,336602,0,3404370,6922737,1188173,0,1188173,3689978,0,30993759,30993759,34683737,925378,70000,82089,152089,1077467,33606270,Sydney Missionary & Bible College,Sydney Missionary & Bible College,Continuing education,N,N,General community in Australia,Croydon,NSW,2132,Australia,03/12/2012,NaN,Large,9,31-Dec,NaN,Advancing_Religion,NSW
2,11000047950,Sydney Missionary & Bible College,Large,n,y,NaN,NaN,n,n,n,NaN,"Conducting an interdenominational, community b...",37,11,20,56.45,3,NaN,01/01/2020,31/12/2020,1153000,1004051,5912764,3975,1668,8075458,35452,8110910,3178458,3307,336602,0,3404370,6922737,1188173,0,1188173,3689978,0,30993759,30993759,34683737,925378,70000,82089,152089,1077467,33606270,Sydney Missionary & Bible College,Sydney Missionary & Bible College,Continuing education,N,N,Overseas communities or charities,Croydon,NSW,2132,Australia,03/12/2012,NaN,Large,9,31-Dec,NaN,Advancing_Education,NSW
3,11000047950,Sydney Missionary & Bible College,Large,n,y,NaN,NaN,n,n,n,NaN,"Conducting an interdenominational, community b...",37,11,20,56.45,3,NaN,01/01/2020,31/12/2020,1153000,1004051,5912764,3975,1668,8075458,35452,8110910,3178458,3307,336602,0,3404370,6922737,1188173,0,1188173,3689978,0,30993759,30993759,34683737,925378,70000,82089,152089,1077467,33606270,Sydney Missionary & Bible College,Sydney Missionary & Bible College,Continuing education,N,N,Overseas communities or charities,Croydon,NSW,2132,Australia,03/12/2012,NaN,Large,9,31-Dec,NaN,Advancing_Religion,NSW
4,11000047950,Sydney Missionary & Bible College,Large,n,y,NaN,NaN,n,n,n,NaN,"Conducting an interdenominational, community b...",37,11,20,56.45,3,NaN,01/01/2020,31/12/2020,1153000,1004051,5912764,397

In [311]:
print(sorted(finaldata.columns)) #Print the column names in sorted format to identify duplicate columns

['ABN', 'Beneficiary', 'Beneficiary Type', 'Charity Name', 'Charity_Legal_Name', 'Charity_Size', 'Classification', 'Country', 'Date_Organisation_Established', 'Financial_Year_End', 'Number_of_Responsible_Persons', 'Operating State', 'Operating online', 'Operating overseas', 'Operating_Countries', 'Other_Organisation_Names', 'Postcode', 'Program name', 'Registration Status', 'Registration_Date', 'State', 'Sub Type', 'Town_City', 'Value_x', 'Value_x', 'Value_y', 'Value_y', 'all other expenses', 'all other revenue', 'basic religious charity', 'cash or accrual', 'charity name', 'charity size', 'conducted activities', 'donations and bequests', 'employee expenses', 'fin report from', 'fin report to', 'grants and donations made for use in Australia', 'grants and donations made for use outside Australia', 'how purposes were pursued', 'interest expenses', 'international activities details', 'international activities undertaken - operating overseas including programs', 'international activities 

In [113]:
finaldata.to_csv('finaldata.csv', index=False)  # Save as CSV
# or
#finaldata.to_excel('finaldata.xlsx', index=False)  # Save as Excel


### We will use finaldata.csv as the source for our tableu operations. Let's do our further analysis over there.

I'm saving the individual files, which I can use to validate data.

In [ ]:
main_final.to_csv('main_final.csv', index=False)  # Save as CSV

In [ ]:
programs_final.to_csv('programs_final.csv', index=False)  # Save as CSV

In [21]:
ais20_final.to_csv('ais20_final.csv', index=False)  # Save as CSV

<h2 style="text-align: center;">-- End --</h2>
